# 08 高度な実験：階層別IC分析と分布の可視化

本ノートブックでは、以下の3つの変更がモデルの予測精度（IC）に与える影響を詳細に分析します。
1. **Days_since_first_seen** (Exp-E)
2. **Absolute_Meeting_ID** (Exp-D)
3. **バタフライ目的変数変換** (Exp-F)

平均的なICだけでなく、その分布を確認することで、予測の安定性を検証します。

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from src.processing import load_and_clean_data
from src.features import generate_features
from src.pooling import pool_boj_data, pool_boj_butterfly, butterfly_to_spread
from src.modeling import walk_forward_validation, calculate_metrics

plt.style.use("ggplot")
EXCEL_PATH = "../data/BOJ_data.xlsx"
MEETING_CSV_PATH = "../data/BOJ_meeting_history.csv"
START_DATE = "2024-01-01"

df_raw = load_and_clean_data(EXCEL_PATH, MEETING_CSV_PATH)

## 分析用共通関数の定義

In [ ]:
def calculate_stratified_ic(res_df, df_input):
    """
    予測結果に対し、DTMグループ、Meeting_Index、FoldごとのICを計算する
    """
    # DTMをマージ (df_inputから取得)
    res = pd.merge(res_df, df_input[["Date", "Meeting_Index", "Days_to_MPM"]].drop_duplicates(), on=["Date", "Meeting_Index"], how="left")
    
    # DTMをグループ化
    res["DTM_Group"] = pd.cut(res["Days_to_MPM"], bins=[-1, 5, 20, 100], labels=["0-5d", "6-20d", "21d+"])
    
    def get_ic(x):
        if len(x) < 2: return np.nan
        return spearmanr(x["Actual"], x["Pred"])[0]
    
    ic_by_dtm = res.groupby("DTM_Group", observed=True).apply(get_ic)
    ic_by_idx = res.groupby("Meeting_Index").apply(get_ic)
    
    # 日付ごとのIC分布を計算（より細かい粒度での安定性確認）
    ic_dist = res.groupby(["Fold", "Date"]).apply(get_ic).reset_index(name="IC")
    
    return {"DTM": ic_by_dtm, "Index": ic_by_idx, "IC_Dist": ic_dist, "Raw": res}

def plot_comparison(base_stats, exp_stats, title):
    fig = plt.figure(figsize=(20, 10))
    gs = fig.add_gridspec(2, 2)
    
    # 1. DTM別比較 (Bar)
    ax1 = fig.add_subplot(gs[0, 0])
    dtm_df = pd.DataFrame({"Baseline": base_stats["DTM"], "Experiment": exp_stats["DTM"]})
    dtm_df.plot(kind="bar", ax=ax1)
    ax1.set_title("IC by Days to MPM")
    ax1.set_ylabel("IC")
    
    # 2. 限月別比較 (Line)
    ax2 = fig.add_subplot(gs[0, 1])
    idx_df = pd.DataFrame({"Baseline": base_stats["Index"], "Experiment": exp_stats["Index"]})
    idx_df.plot(kind="line", marker="o", ax=ax2)
    ax2.set_title("IC by Meeting Index (M1-M8)")
    ax2.set_xticks(range(1, 9))
    ax2.set_ylabel("IC")
    
    # 3. IC分布比較 (Boxplot)
    ax3 = fig.add_subplot(gs[1, :])
    dist_df = pd.concat([
        base_stats["IC_Dist"].assign(Model="Baseline"),
        exp_stats["IC_Dist"].assign(Model="Experiment")
    ])
    sns.boxplot(x="Fold", y="IC", hue="Model", data=dist_df, ax=ax3)
    ax3.set_title("IC Distribution by Fold (Stability Analysis)")
    
    plt.suptitle(title, fontsize=18)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

## データの準備 (Baseline)

In [ ]:
df_feat_base = generate_features(df_raw)
df_pooled_base = pool_boj_data(df_feat_base).drop(columns=["Absolute_Meeting_ID", "Days_since_first_seen"])

res_3d_base = walk_forward_validation(df_pooled_base, "Target_3d", START_DATE)
res_5d_base = walk_forward_validation(df_pooled_base, "Target_5d", START_DATE)

stats_3d_base = calculate_stratified_ic(res_3d_base, df_pooled_base)
stats_5d_base = calculate_stratified_ic(res_5d_base, df_pooled_base)

## 分析1: Days_since_first_seen (Exp-E) の影響

In [ ]:
df_pooled_e = pool_boj_data(df_feat_base).drop(columns=["Absolute_Meeting_ID"])
res_3d_e = walk_forward_validation(df_pooled_e, "Target_3d", START_DATE)
res_5d_e = walk_forward_validation(df_pooled_e, "Target_5d", START_DATE)

stats_3d_e = calculate_stratified_ic(res_3d_e, df_pooled_e)
stats_5d_e = calculate_stratified_ic(res_5d_e, df_pooled_e)

plot_comparison(stats_3d_base, stats_3d_e, "Exp-E (Days_since_first_seen) vs Baseline")

## 分析2: Absolute_Meeting_ID (Exp-D) の影響

In [ ]:
df_pooled_d = pool_boj_data(df_feat_base).drop(columns=["Days_since_first_seen"])
res_3d_d = walk_forward_validation(df_pooled_d, "Target_3d", START_DATE)
res_5d_d = walk_forward_validation(df_pooled_d, "Target_5d", START_DATE)

stats_3d_d = calculate_stratified_ic(res_3d_d, df_pooled_d)
stats_5d_d = calculate_stratified_ic(res_5d_d, df_pooled_d)

plot_comparison(stats_3d_base, stats_3d_d, "Exp-D (Absolute_Meeting_ID) vs Baseline")

## 分析3: バタフライ目的変数変換 (Exp-F) の影響

In [ ]:
def get_butterfly_results(df_raw, horizon):
    df_feat = generate_features(df_raw)
    df_b = pool_boj_butterfly(df_feat)
    target_col = f"Target_{horizon}d_B_norm"
    df_input = df_b.rename(columns={"Butterfly_Index": "Meeting_Index"})
    
    res_b = walk_forward_validation(df_input, target_col, START_DATE)
    res_b = pd.merge(res_b, df_input[["Date", "Meeting_Index", f"Target_{horizon}d_B_std"]], on=["Date", "Meeting_Index"], how="left")
    
    # 逆変換ロジック
    pred_pivot = res_b.pivot(index="Date", columns="Meeting_Index", values="Pred")
    std_pivot = res_b.pivot(index="Date", columns="Meeting_Index", values=f"Target_{horizon}d_B_std")
    pred_raw_b = pred_pivot * std_pivot
    
    reconstructed_rows = []
    for date, row in pred_raw_b.iterrows():
        s_diff = butterfly_to_spread(row.values)
        for i, val in enumerate(s_diff):
            reconstructed_rows.append({"Date": date, "Meeting_Index": i+1, "Pred_Recon": val})
    
    pred_recon = pd.DataFrame(reconstructed_rows)
    
    # Fold情報とDTM情報を元の res_b から取得してマージ
    fold_dtm = res_b[["Date", "Fold"]].drop_duplicates()
    pred_recon = pd.merge(pred_recon, fold_dtm, on="Date", how="left")
    
    # 正解データを pool_boj_data から取得
    df_p = pool_boj_data(df_feat)
    final_res = pd.merge(
        pred_recon, 
        df_p[df_p["Is_Tenor_OIS"]==0][["Date", "Meeting_Index", f"Target_{horizon}d"]], 
        on=["Date", "Meeting_Index"], 
        how="inner"
    ).rename(columns={f"Target_{horizon}d": "Actual", "Pred_Recon": "Pred"})
    
    return final_res

res_3d_f = get_butterfly_results(df_raw, 3)
res_5d_f = get_butterfly_results(df_raw, 5)

stats_3d_f = calculate_stratified_ic(res_3d_f, df_pooled_base)
stats_5d_f = calculate_stratified_ic(res_5d_f, df_pooled_base)

plot_comparison(stats_3d_base, stats_3d_f, "Exp-F (Butterfly Target) vs Baseline")